# 보관용 공개 사본

당시 팀 프로젝트의 구현 자료입니다. 출력과 개인 공유 경로를 정리했습니다. 데이터·학습 가중치는 포함하지 않으며, 현재 환경에서 전체 실행을 검증하지 않았습니다. 상세한 버전 차이와 실행 조건은 `docs/evidence.md`, `docs/reproduction.md`를 참고하세요.


In [ ]:
# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

# 필요한 라이브러리 설치
!pip install -q datasets transformers

In [ ]:
from datasets import load_dataset
import pandas as pd

# Smilegate Unsmile 데이터셋 로드
dataset = load_dataset("smilegate-ai/kor_unsmile")

# Train과 Valid 데이터 하나로 합치기
raw_all_df = pd.concat([pd.DataFrame(dataset['train']), pd.DataFrame(dataset['valid'])], ignore_index=True)

print(f"전체 Raw 데이터 개수: {len(raw_all_df)}")
raw_all_df.head()

In [ ]:
import re
import numpy as np

# 1. 욕설/비속어 정규식 키워드 리스트
PROFANE_KEYWORDS = [
    '시발', '씨발', '씨바', 'ㅅㅂ', '씨빨', '슈발', '쉬발',
    '개새끼', '개새', '새끼', '색기', '쌔끼', '쉐끼', 'ㅅㄲ',
    '병신', '붕신', '빙신', 'ㅂㅅ', '병싱',
    '지랄', '지럴', 'ㅈㄹ', '찌랄',
    '좆같', '좆까', '좆나', '좆망', '조까', '존나', '존내', '졸라', 'ㅈㄴ',
    '개같', '개년', '개놈', '개소리', '개짓', '개판',
    '미친놈', '미친년', '미친새', '또라이', '돌아이',
    '꺼져', '닥쳐', '닥치',
    '엿먹', '뒤져', '뒈져', '디져', '뒈짐',
    '멍청', '떨빵', '한심',
    '틀딱', '급식충', '맘충', '한남충', '된장녀', '김치녀',
    '앰창', '엠창', '느금', '니미', '니애미', '패드립',
    '창녀', '걸레', '창놈',
    '붕딱', '등신', '개돼지', '쓰레기', '토나와', '역겨워', '극혐'
]

pattern = '|'.join(map(re.escape, PROFANE_KEYWORDS))

def transform_unsmile_dataset_v3(df):
    """
    [v3 개편 반영]
    1. TARGET: 8개 집단 혐오 카테고리 전체 합산
    2. PROFANE: 원본 '악플/욕설' 컬럼 OR 키워드 정규식
    3. bias_level / severity: 강 -> 약 순서 조건문 (cond_3 -> cond_2 -> cond_1 -> cond_0)
    """
    cats8 = ['여성/가족', '남성', '성소수자', '인종/국적', '연령', '지역', '종교', '기타 혐오']

    # 1. TARGET (집단 혐오 유무)
    df['TARGET'] = (df[cats8].sum(axis=1) > 0).astype(int)

    # 2. PROFANE (욕설 유무: 원본 악플/욕설 OR 정규식 패턴)
    regex_hit = df['문장'].astype(str).str.contains(pattern, regex=True)
    df['PROFANE'] = regex_hit.astype(int)

    # 3. 강 -> 약 우선순위 조건 정의
    cond_3 = (df['TARGET'] == 1) & (df['PROFANE'] == 1)  # 집단 + 욕설
    cond_2 = (df['TARGET'] == 1) & (df['PROFANE'] == 0)  # 집단만
    cond_1 = (df['TARGET'] == 0) & (df['PROFANE'] == 1)  # 욕설만
    cond_0 = (df['TARGET'] == 0) & (df['PROFANE'] == 0) & (df['clean'] == 1)  # clean

    conditions = [cond_3, cond_2, cond_1, cond_0]

    # bias_level (0~3) & severity (0~2) 생성
    df['bias_level'] = np.select(conditions, [3, 2, 1, 0], default=-1)
    df['severity']   = np.select(conditions, [2, 1, 1, 0], default=-1)

    # 4. default=-1 처리 (찌꺼기 데이터 filtering)
    filtered_df = df[df['bias_level'] != -1].copy()

    # 5. 스키마 구성
    new_df = pd.DataFrame()
    new_df['text'] = filtered_df['문장'].astype(str)
    new_df['sycophancy'] = 0
    new_df['bias_level'] = filtered_df['bias_level'].astype(int)
    new_df['severity'] = filtered_df['severity'].astype(int)

    return new_df

# 전처리 함수 실행
final_processed_df = transform_unsmile_dataset_v3(raw_all_df)

print(f"필터링 후 남아있는 총 데이터 개수: {len(final_processed_df)}")
print("\n--- v3 bias_level 라벨 분포 ---")
print(final_processed_df['bias_level'].value_counts().sort_index())

In [ ]:
import os
from sklearn.model_selection import train_test_split

# 7 : 1.5 : 1.5 비율 분할
train_df, test_valid_df = train_test_split(final_processed_df, test_size=0.3, random_state=42)
valid_df, test_df = train_test_split(test_valid_df, test_size=0.5, random_state=42)

# 저장할 구글 드라이브 폴더 경로 설정
save_path = '/content/drive/MyDrive/voda-sentinel/data/processed/'

# 폴더가 없으면 생성
os.makedirs(save_path, exist_ok=True)

# v3파일명으로 CSV 저장
train_df.to_csv(save_path + 'train_processed_v3.csv', index=False, encoding='utf-8-sig')
valid_df.to_csv(save_path + 'valid_processed_v3.csv', index=False, encoding='utf-8-sig')
test_df.to_csv(save_path + 'test_processed_v3.csv', index=False, encoding='utf-8-sig')

print("="*50)
print("🎉 v3 버전 데이터 전처리 및 저장 완료!")
print(f"저장 위치: {save_path}")
print("생성된 파일: train_processed_v3.csv, valid_processed_v3.csv, test_processed_v3.csv")
print("="*50)